# 🧹 전역 중복 제거 및 데이터 정제 작업

이 노트북은 `parquet_logs` 폴더 내의 모든 데이터를 읽어와서 **전역 중복(Global Duplicates)**을 제거한 뒤, 다시 월별 파케이 파일로 저장합니다.

### 핵심 로직
1. 모든 파케이 파일을 `Lazy` 모드로 스캔합니다.
2. 모든 컬럼의 값이 완벽히 일치하는 행을 제거합니다 (`unique`).
3. 정제된 데이터를 다시 `YYYY-MM` 기준으로 파티셔닝하여 저장합니다.

In [ ]:
import polars as pl
import glob
import os

# 1. 설정
input_dir = "parquet_logs"
output_dir = "parquet_logs_cleaned"
os.makedirs(output_dir, exist_ok=True)

print("🔍 데이터 스캔 중...")
files = sorted(glob.glob(f"{input_dir}/log_*.parquet"))

# 2. Lazy 스캔 (계획 수립)
lf = pl.scan_parquet(files)

print(f"📊 분석 대상 파일 수: {len(files)}개")

# 3. 중복 제거 전후 카운트 비교
total_count = lf.select(pl.len()).collect().item()
print(f"- 중복 제거 전 총 건수: {total_count:,}건")

print("✨ 중복 제거 작업 시작 (전체 컬럼 기준)...")
df_cleaned = lf.unique().collect() # 이 단계에서 실제 연산 수행

cleaned_count = len(df_cleaned)
duplicate_count = total_count - cleaned_count

print(f"- 중복 제거 후 총 건수: {cleaned_count:,}건")
print(f"- 제거된 중복 데이터: {duplicate_count:,}건 (약 {duplicate_count/total_count:.4%})")

# 4. 다시 월별로 쪼개서 저장
print("\n🚀 정제된 데이터를 월별 파일로 재저장합니다...")

# timestamp에서 YYYY-MM 추출 (가장 정확한 기준)
df_cleaned = df_cleaned.with_columns(
    pl.col("timestamp").str.slice(0, 7).alias("year_month")
)

# 파티션별 저장
partitions = df_cleaned.partition_by("year_month", as_dict=True)

for key, part_df in partitions.items():
    ym = key[0] if isinstance(key, tuple) else key
    output_path = f"{output_dir}/log_{ym}.parquet"
    
    # 보조 컬럼 제거 후 저장
    part_df.drop("year_month").write_parquet(output_path)
    print(f"✅ {output_path} 저장 완료")

print("\n🎉 모든 작업이 완료되었습니다! 'parquet_logs_cleaned' 폴더를 확인하세요.")